# t-SNE From Scratch on UCI Optdigits — P00–P13

Exact NumPy implementation, mathematical verification, neighborhood evaluation, multi-seed stability and reproducibility handoff.

**Execution profiles**
- `quick`: P00–P08; safe core audit.
- `screening`: adds P09–P10.
- `full`: adds P11–P12, including exact full 5620-sample run.

In [1]:
from pathlib import Path
import gc, json, time, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EXECUTION_PROFILE = "quick"  # quick | screening | full
assert EXECUTION_PROFILE in {"quick","screening","full"}
RUN_SCREENING = EXECUTION_PROFILE in {"screening","full"}
RUN_FULL = EXECUTION_PROFILE == "full"
print("Execution profile:", EXECUTION_PROFILE)

Execution profile: quick


## P00 — Project architecture

### AI PROMPTING LOG

> Tôi đang xây dựng case study t-SNE from scratch trên UCI Optdigits. Hãy khóa pipeline end-to-end: raw Optdigits → data integrity → squared Euclidean distance → conditional Gaussian similarities → perplexity matching bằng binary search → symmetric P → random 2D Y → Student-t Q → KL(P||Q) → analytic gradient → optimization → mathematical validation → synthetic validation → perplexity sensitivity → multi-scale Trustworthiness/Continuity → multi-seed stability → final full-data run → export. Không dùng sklearn.manifold.TSNE. Labels không tham gia fitting. Mỗi phase phải có assertions/checkpoint. Không chọn perplexity bằng KL thấp nhất và không hard-code winner trước experiment. Logic cần giữ nguyên: input → operation → validation → output.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [2]:
PIPELINE = "raw Optdigits → audit → D → conditional P → perplexity matching → symmetric P → Y → Q → KL → gradient → optimizer → validation → sensitivity → T/C → stability → final run → handoff"
print(PIPELINE)
print("CHECKPOINT 0 — Project Architecture: PASS ✅")

raw Optdigits → audit → D → conditional P → perplexity matching → symmetric P → Y → Q → KL → gradient → optimizer → validation → sensitivity → T/C → stability → final run → handoff
CHECKPOINT 0 — Project Architecture: PASS ✅


## P01 — Load and audit Optdigits

### AI PROMPTING LOG

> Tôi có raw optdigits.tra/optdigits.tes. Hãy load bằng pandas/NumPy, kiểm tra shape 3823×65 và 1797×65, gán 64 tên Pixel_1..Pixel_64 + label, concat thành 5620×65, tách X bằng explicit feature selection và y bằng label. Kiểm tra finite, range 0–16, class distribution, variance. Không scale, không PCA preprocessing. Khi lưu CSV bắt buộc index=False; reload và assert không có Unnamed column và X vẫn 5620×64. Chỉ khi PASS mới sang phase sau.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [3]:
def locate_data_root():
    candidates=[Path.cwd()/"data", Path.cwd().parent/"data", Path("/content/data")]
    for root in candidates:
        if (root/"processed/optdigits_clean.csv").exists(): return root
    raise FileNotFoundError("Run notebook from extracted package so data/ is available.")
DATA_ROOT=locate_data_root(); PACKAGE_ROOT=DATA_ROOT.parent
OUTPUT_ROOT=PACKAGE_ROOT/"outputs"; TABLE_DIR=OUTPUT_ROOT/"tables"; FIGURE_DIR=OUTPUT_ROOT/"figures"; EMBEDDING_DIR=OUTPUT_ROOT/"embeddings"
for d in [TABLE_DIR,FIGURE_DIR,EMBEDDING_DIR]: d.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(DATA_ROOT/"processed/optdigits_clean.csv")
feature_names=[f"Pixel_{i}" for i in range(1,65)]
assert df.shape==(5620,65) and df.columns.tolist()==feature_names+["label"]
assert not any(str(c).startswith("Unnamed") for c in df.columns)
X=df[feature_names].to_numpy(np.float64); y=df["label"].to_numpy(np.int64)
assert X.shape==(5620,64) and np.isfinite(X).all() and X.min()>=0 and X.max()<=16
print(pd.Series(y).value_counts().sort_index())
print("CHECKPOINT 1 — Optdigits Data Integrity: PASS ✅")

0    554
1    571
2    557
3    572
4    568
5    558
6    558
7    566
8    554
9    562
Name: count, dtype: int64
CHECKPOINT 1 — Optdigits Data Integrity: PASS ✅


## P02 — Squared Euclidean distance

### AI PROMPTING LOG

> Từ X sạch 5620×64, tự viết squared_euclidean_distances(X) bằng ||xi||²+||xj||²−2xi·xj, NumPy vectorized, float64. Kiểm tra symmetry, diagonal 0, nonnegative, finite, và so với brute-force trên tiny subset. Không sqrt. Chưa xây P/Q/t-SNE. In memory estimate cho N×N.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [4]:
def squared_euclidean_distances(X):
    X=np.asarray(X,dtype=np.float64); norms=np.sum(X*X,axis=1,keepdims=True)
    D=norms+norms.T-2*(X@X.T); D=0.5*(D+D.T); D=np.maximum(D,0); np.fill_diagonal(D,0); return D
Xt=X[:8]; D=squared_euclidean_distances(Xt); B=np.array([[np.sum((Xt[i]-Xt[j])**2) for j in range(8)] for i in range(8)])
assert np.allclose(D,B,atol=1e-10) and np.allclose(D,D.T) and np.allclose(np.diag(D),0)
X_distance_check=X[:256]; D_check=squared_euclidean_distances(X_distance_check)
print("One full float64 matrix MiB:",5620*5620*8/1024**2)
print("CHECKPOINT 2 — Squared Euclidean Distance: PASS ✅")

One full float64 matrix MiB: 240.9698486328125
CHECKPOINT 2 — Squared Euclidean Distance: PASS ✅


## P03 — Conditional probability, entropy, perplexity

### AI PROMPTING LOG

> Từ một distance row và beta>0, tự viết Gaussian conditional probability p(j|i), đặt self probability=0, dùng numerically-stable normalization. Tính Shannon entropy bằng natural log và perplexity=exp(H). Kiểm tra sum=1, range, beta tăng → entropy/perplexity không tăng. Chưa binary-search beta.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [5]:
def entropy_and_probabilities(distance_row,beta,self_index):
    d=np.asarray(distance_row,dtype=np.float64); mask=np.ones(len(d),bool); mask[self_index]=False
    z=-beta*np.maximum(d[mask],0); z-=z.max(); w=np.exp(z); p=np.zeros(len(d)); p[mask]=w/w.sum(); pos=p>0
    H=float(-np.sum(p[pos]*np.log(p[pos]))); return p,H,float(np.exp(H))
d=D_check[0]; base=1/max(np.median(np.delete(d,0)),1e-12)
vals=[entropy_and_probabilities(d,b,0) for b in [base*.25,base,base*4]]
assert all(np.isclose(v[0].sum(),1) and v[0][0]==0 for v in vals)
assert vals[0][2]>=vals[1][2]>=vals[2][2]
print("CHECKPOINT 3 — Conditional Probability & Perplexity: PASS ✅")

CHECKPOINT 3 — Conditional Probability & Perplexity: PASS ✅


## P04 — Perplexity matching and P

### AI PROMPTING LOG

> Tự viết binary_search_beta để tìm beta_i sao cho entropy gần log(target perplexity), adaptive bounds không dùng scipy. Sau đó xây P_cond row-wise và P=(P_cond+P_cond.T)/(2N). Kiểm tra achieved perplexity, P_cond row sums, P symmetry, diagonal 0, sum(P)=1. Không dùng heuristic 3*perplexity<n như ràng buộc toán học.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [6]:
def binary_search_beta(distance_row,target_perplexity,self_index,tol=1e-5,max_iter=60):
    n=len(distance_row); assert 1<target_perplexity<=n-1; target=np.log(target_perplexity); beta=1.; lo=-np.inf; hi=np.inf
    for it in range(1,max_iter+1):
        used=beta; p,H,perp=entropy_and_probabilities(distance_row,used,self_index); err=H-target
        if abs(err)<=tol: return {"probabilities":p,"beta":used,"entropy":H,"perplexity":perp,"iterations":it,"converged":True}
        if err>0: lo=used; beta=used*2 if np.isinf(hi) else .5*(used+hi)
        else: hi=used; beta=used/2 if np.isinf(lo) else .5*(used+lo)
    return {"probabilities":p,"beta":used,"entropy":H,"perplexity":perp,"iterations":it,"converged":False}
def conditional_probability_matrix(D,target_perplexity,tol=1e-5,max_iter=60):
    n=D.shape[0]; P=np.zeros_like(D); bet=np.zeros(n); per=np.zeros(n); ent=np.zeros(n); its=np.zeros(n,int); conv=np.zeros(n,bool)
    for i in range(n):
        r=binary_search_beta(D[i],target_perplexity,i,tol,max_iter); P[i]=r["probabilities"]; bet[i]=r["beta"]; per[i]=r["perplexity"]; ent[i]=r["entropy"]; its[i]=r["iterations"]; conv[i]=r["converged"]
    return {"P_cond":P,"betas":bet,"perplexities":per,"entropies":ent,"iterations":its,"converged":conv}
def symmetrize_probabilities(Pc):
    P=(Pc+Pc.T)/(2*len(Pc)); np.fill_diagonal(P,0); return P
cr=conditional_probability_matrix(D_check,30); assert cr["converged"].all(); P_cond_check=cr["P_cond"]; P_check=symmetrize_probabilities(P_cond_check)
assert np.allclose(P_cond_check.sum(1),1,atol=1e-10) and np.allclose(P_check,P_check.T) and np.isclose(P_check.sum(),1)
print("CHECKPOINT 4 — Perplexity Matching & Symmetric P: PASS ✅")

CHECKPOINT 4 — Perplexity Matching & Symmetric P: PASS ✅


## P05 — Student-t Q and KL

### AI PROMPTING LOG

> Khởi tạo Y 2D reproducible bằng NumPy; xây Student-t numerator=(1+||yi-yj||²)^−1, diagonal 0, normalize thành Q. Tự viết KL(P||Q), chỉ tính nơi P>0 và lỗi nếu Q<=0 ở đó. Kiểm tra Q symmetry/sum, KL(P||P)=0, translation invariance. Chưa gradient/update.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [7]:
def initialize_embedding(n_samples,n_components=2,random_state=42,scale=1e-4):
    return np.random.default_rng(random_state).normal(0,scale,size=(n_samples,n_components)).astype(np.float64)
def low_dimensional_affinities(Y):
    D=squared_euclidean_distances(Y); num=1/(1+D); np.fill_diagonal(num,0); Q=num/num.sum(); return Q,num,D
def kl_divergence(P,Q):
    m=P>0; assert np.all(Q[m]>0); return float(np.sum(P[m]*(np.log(P[m])-np.log(Q[m]))))
Y_check=initialize_embedding(len(P_check)); Q_check,Q_numerator_check,D_low_check=low_dimensional_affinities(Y_check); initial_kl_check=kl_divergence(P_check,Q_check)
assert np.isclose(Q_check.sum(),1) and np.allclose(Q_check,Q_check.T) and np.isclose(kl_divergence(P_check,P_check),0,atol=1e-12)
Q_shift,_,_=low_dimensional_affinities(Y_check+np.array([5.,-3.])); assert np.allclose(Q_check,Q_shift,atol=1e-12)
print("CHECKPOINT 5 — Low-Dimensional Q & KL Divergence: PASS ✅")

CHECKPOINT 5 — Low-Dimensional Q & KL Divergence: PASS ✅


## P06 — Gradient verification

### AI PROMPTING LOG

> Tự viết analytic t-SNE gradient 4 sum_j (Pij-Qij)(yi-yj)/(1+||yi-yj||²), vectorized. Viết brute-force reference và central finite-difference KL numerical gradient trên tiny dataset được xây lại đúng P. So sánh analytic vs brute-force và numerical; kiểm tra total gradient ≈0. Chưa optimizer.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [8]:
def tsne_gradient(P,Q,numerator,Y):
    A=(P-Q)*numerator; return 4*(A.sum(1,keepdims=True)*Y-A@Y)
def kl_from_embedding(P,Y):
    Q,_,_=low_dimensional_affinities(Y); return kl_divergence(P,Q)
# independent tiny problem
Xtiny=X[:8]; Dtiny=squared_euclidean_distances(Xtiny); crt=conditional_probability_matrix(Dtiny,3,tol=1e-7,max_iter=80); Pt=symmetrize_probabilities(crt["P_cond"]); Yt=initialize_embedding(8,2,123,.1); Qt,numt,_=low_dimensional_affinities(Yt); gt=tsne_gradient(Pt,Qt,numt,Yt)
eps=1e-6; gn=np.zeros_like(Yt)
for i in range(8):
    for d in range(2):
        yp=Yt.copy(); ym=Yt.copy(); yp[i,d]+=eps; ym[i,d]-=eps; gn[i,d]=(kl_from_embedding(Pt,yp)-kl_from_embedding(Pt,ym))/(2*eps)
rel=np.linalg.norm(gt-gn)/max(np.linalg.norm(gt)+np.linalg.norm(gn),1e-12); assert rel<1e-4 and np.allclose(gt.sum(0),0,atol=1e-12)
print("Finite-difference relative error:",rel)
print("CHECKPOINT 6 — t-SNE Gradient Verification: PASS ✅")

Finite-difference relative error: 1.4483071348660293e-09
CHECKPOINT 6 — t-SNE Gradient Verification: PASS ✅


## P07 — Optimization engine

### AI PROMPTING LOG

> Dùng gradient đã verify để xây optimize_tsne_embedding và class TSNEFromScratch với learning rate, early exaggeration, momentum, adaptive gains, recentering, KL history, best_Y restoration, early stopping. True KL luôn dùng P gốc; P_work exaggeration không renormalize. Best state phải lưu Y_current tương ứng exact KL, không suy ngược Y-update. Không cung cấp transform(X_new). Functional test trên subset, không gọi perplexity=30 là tối ưu.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [9]:
def optimize_tsne_embedding(P,n_components=2,learning_rate=200,n_iter=350,early_exaggeration=12,early_exaggeration_iter=100,initial_momentum=.5,final_momentum=.8,momentum_switch_iter=100,min_gain=.01,patience=80,min_kl_improvement=1e-7,random_state=42,init_scale=1e-4,verbose=False):
    Y=initialize_embedding(len(P),n_components,random_state,init_scale); prev=np.zeros_like(Y); gains=np.ones_like(Y); hist=[]; gh=[]; best=np.inf; bestY=None; bestit=None; noimp=0; stopped=False; Q0,_,_=low_dimensional_affinities(Y); init=kl_divergence(P,Q0)
    for it in range(1,n_iter+1):
        Ycur=Y.copy(); Q,num,_=low_dimensional_affinities(Ycur); kl=kl_divergence(P,Q); hist.append(kl); ex=it<=early_exaggeration_iter; grad=tsne_gradient(P*early_exaggeration if ex else P,Q,num,Ycur); gh.append(np.linalg.norm(grad))
        if not ex:
            if kl<best-min_kl_improvement: best=kl; bestY=Ycur.copy(); bestit=it; noimp=0
            else: noimp+=1
            if patience and noimp>=patience: stopped=True; break
        mom=initial_momentum if it<momentum_switch_iter else final_momentum; changed=np.sign(grad)!=np.sign(prev); gains=np.where(changed,gains+.2,gains*.8); gains=np.maximum(gains,min_gain); upd=mom*prev-learning_rate*gains*grad; Y=Ycur+upd; Y-=Y.mean(0,keepdims=True); prev=upd
        if not np.isfinite(Y).all(): raise FloatingPointError("Embedding diverged")
        if verbose and (it==1 or it%50==0 or it==early_exaggeration_iter): print(f"iter={it} KL={kl:.6f} grad={gh[-1]:.6f}")
    Qlast,_,_=low_dimensional_affinities(Y); last=kl_divergence(P,Qlast)
    if it>early_exaggeration_iter and last<best: best=last; bestY=Y.copy(); bestit=it
    if bestY is None: bestY=Y.copy(); best=last; bestit=it
    Qb,_,_=low_dimensional_affinities(bestY); restored=kl_divergence(P,Qb)
    return {"embedding":bestY,"initial_kl":init,"best_kl_divergence":best,"kl_divergence":restored,"last_kl_before_restore":last,"best_iteration":bestit,"n_iter":it,"stopped_early":stopped,"kl_history":np.asarray(hist),"gradient_norm_history":np.asarray(gh)}
class TSNEFromScratch:
    def __init__(self,perplexity=30,random_state=42,n_iter=350,**kwargs): self.perplexity=perplexity; self.random_state=random_state; self.n_iter=n_iter; self.kwargs=kwargs
    def fit(self,X):
        X=np.asarray(X,np.float64); D=squared_euclidean_distances(X); cr=conditional_probability_matrix(D,self.perplexity); assert cr["converged"].all(); P=symmetrize_probabilities(cr["P_cond"]); o=optimize_tsne_embedding(P,n_iter=self.n_iter,random_state=self.random_state,**self.kwargs)
        self.embedding_=o["embedding"]; self.initial_kl_divergence_=o["initial_kl"]; self.kl_divergence_=o["kl_divergence"]; self.best_kl_divergence_=o["best_kl_divergence"]; self.best_iteration_=o["best_iteration"]; self.n_iter_=o["n_iter"]; self.stopped_early_=o["stopped_early"]; self.kl_history_=o["kl_history"]; self.gradient_norm_history_=o["gradient_norm_history"]; self.achieved_perplexities_=cr["perplexities"]; self.perplexity_search_iterations_=cr["iterations"]; self.betas_=cr["betas"]; self.is_fitted_=True; return self
    def fit_transform(self,X): return self.fit(X).embedding_.copy()
# functional test only; p=30 is not claimed optimal
model=TSNEFromScratch(perplexity=30,random_state=42,n_iter=250,early_exaggeration_iter=80,momentum_switch_iter=80,patience=70); Z=model.fit_transform(X[:256]); assert Z.shape==(256,2) and np.isfinite(Z).all() and model.kl_divergence_<model.initial_kl_divergence_
print("CHECKPOINT 7 — t-SNE Optimization Engine: PASS ✅")

CHECKPOINT 7 — t-SNE Optimization Engine: PASS ✅


## P08 — Independent synthetic validation

### AI PROMPTING LOG

> Tạo 3 Gaussian clusters high-dimensional bằng NumPy, fit TSNEFromScratch chỉ với X, labels chỉ post-hoc. Tự xây kNN overlap và local label purity. So sánh observed overlap với random baseline; không dùng scatter đẹp làm bằng chứng duy nhất, không diễn giải global inter-cluster distance.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [10]:
def knn_idx(D,k):
    W=D.copy(); np.fill_diagonal(W,np.inf); return np.argsort(W,axis=1,kind="mergesort")[:,:k]
def overlap(a,b):
    return np.mean([len(set(x).intersection(z))/a.shape[1] for x,z in zip(a,b)])
rng=np.random.default_rng(2026); centers=np.zeros((3,12)); centers[1,:6]=6; centers[1,6:]=-2; centers[2,:6]=-3; centers[2,6:]=6
Xs=np.vstack([rng.normal(centers[c],.8,size=(60,12)) for c in range(3)]); ys=np.repeat(np.arange(3),60); perm=rng.permutation(180); Xs=Xs[perm]; ys=ys[perm]
ms=TSNEFromScratch(perplexity=30,random_state=42,n_iter=300,early_exaggeration_iter=80,momentum_switch_iter=80,patience=80); Zs=ms.fit_transform(Xs); kh=knn_idx(squared_euclidean_distances(Xs),10); kl=knn_idx(squared_euclidean_distances(Zs),10); ov=overlap(kh,kl); base=10/179
purity=np.mean([np.mean(ys[n]==ys[i]) for i,n in enumerate(kl)]); assert ov>3*base and purity>.8
print({"overlap":ov,"random_baseline":base,"lowD_purity":purity})
print("CHECKPOINT 8 — Independent Synthetic Validation: PASS ✅")

{'overlap': np.float64(0.47611111111111115), 'random_baseline': 0.055865921787709494, 'lowD_purity': np.float64(1.0)}
CHECKPOINT 8 — Independent Synthetic Validation: PASS ✅


## P09 — Perplexity sensitivity

### AI PROMPTING LOG

> Tạo deterministic stratified Optdigits screening subset; giữ dataset/optimizer/seed cố định, chỉ thay perplexity [5,10,20,30,40,50]. Lưu embedding, runtime, iteration, KL diagnostics và achieved perplexity. Không rank bằng KL và chưa chọn best. Tạo visual comparison chỉ post-hoc.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [11]:
PERPLEXITY_GRID=[5,10,20,30,40,50]
def stratified_sample_indices(y,n_total=1000,random_state=2026):
    rng=np.random.default_rng(random_state); labels=np.unique(y); q,r=divmod(n_total,len(labels)); out=[]
    for pos,label in enumerate(labels): out.extend(rng.choice(np.where(y==label)[0],q+(pos<r),replace=False).tolist())
    return rng.permutation(np.asarray(out,int))
screen_indices=stratified_sample_indices(y,1000); X_screen=X[screen_indices]; y_screen=y[screen_indices]
pd.DataFrame({"original_index":screen_indices,"label":y_screen}).to_csv(TABLE_DIR/"optdigits_screening_indices.csv",index=False)
if RUN_SCREENING:
    screening_embeddings={}; rows=[]
    for p in PERPLEXITY_GRID:
        t=time.perf_counter(); m=TSNEFromScratch(perplexity=p,random_state=42,n_iter=750,early_exaggeration_iter=250,momentum_switch_iter=250,patience=120); Z=m.fit_transform(X_screen); runtime=time.perf_counter()-t
        assert Z.shape==(1000,2) and np.isfinite(Z).all() and m.best_kl_divergence_<m.initial_kl_divergence_
        screening_embeddings[p]=Z.copy(); err=np.max(np.abs(m.achieved_perplexities_-p))
        rows.append({"Perplexity":p,"Samples":len(X_screen),"Initial KL":m.initial_kl_divergence_,"Best KL":m.best_kl_divergence_,"Final Restored KL":m.kl_divergence_,"Best Iteration":m.best_iteration_,"Actual Iterations":m.n_iter_,"Stopped Early":m.stopped_early_,"Runtime Seconds":runtime,"Achieved Perplexity Min":m.achieved_perplexities_.min(),"Achieved Perplexity Mean":m.achieved_perplexities_.mean(),"Achieved Perplexity Max":m.achieved_perplexities_.max(),"Max Perplexity Abs Error":err})
        pd.DataFrame({"original_index":screen_indices,"TSNE_1":Z[:,0],"TSNE_2":Z[:,1],"label":y_screen}).to_csv(EMBEDDING_DIR/f"optdigits_tsne_perplexity_{p}.csv",index=False)
        gc.collect()
    screening_results_df=pd.DataFrame(rows).sort_values("Perplexity").reset_index(drop=True); screening_results_df.to_csv(TABLE_DIR/"perplexity_screening_summary.csv",index=False); display(screening_results_df)
    fig,axes=plt.subplots(2,3,figsize=(15,10)); axes=axes.ravel()
    for ax,p in zip(axes,PERPLEXITY_GRID):
        Z=screening_embeddings[p]; ax.scatter(Z[:,0],Z[:,1],c=y_screen,s=8); ax.set_title(f"Perplexity={p}"); ax.grid(alpha=.15)
    fig.suptitle("Optdigits — t-SNE Perplexity Sensitivity"); plt.tight_layout(); plt.savefig(FIGURE_DIR/"optdigits_perplexity_grid.png",dpi=170,bbox_inches="tight"); plt.show()
    print("CHECKPOINT 9 — Perplexity Sensitivity Study: PASS ✅")
else:
    print("P09 READY — set EXECUTION_PROFILE='screening' or 'full' to execute heavy grid.")

P09 READY — set EXECUTION_PROFILE='screening' or 'full' to execute heavy grid.


## P10 — Trustworthiness and Continuity

### AI PROMPTING LOG

> Tự viết multi-scale Trustworthiness và Continuity ở k=[5,10,20,50] từ neighbor ranks, không dùng sklearn metric. Identity geometry phải cho T=C=1. Tạo quality table và Pareto shortlist; không weighted score tùy ý và chưa kết luận global optimum.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [12]:
def neighbor_order_and_ranks(D):
    W=D.copy(); np.fill_diagonal(W,np.inf); order=np.argsort(W,axis=1,kind="mergesort"); n=len(D); ranks=np.empty((n,n),np.int32); ranks[np.arange(n)[:,None],order]=np.arange(1,n+1)[None,:]; return order,ranks
def trustworthiness_from_ranks(ho,hr,lo,k):
    n=len(ho); assert 1<=k<n/2; r=hr[np.arange(n)[:,None],lo[:,:k]]; pen=np.sum((r-k)*(r>k)); return float(1-2*pen/(n*k*(2*n-3*k-1)))
def continuity_from_ranks(ho,lo,lr,k):
    n=len(ho); assert 1<=k<n/2; r=lr[np.arange(n)[:,None],ho[:,:k]]; pen=np.sum((r-k)*(r>k)); return float(1-2*pen/(n*k*(2*n-3*k-1)))
def neighborhood_quality(Z,ho,hr,k_values=(5,10,20,50)):
    lo,lr=neighbor_order_and_ranks(squared_euclidean_distances(Z)); out={}
    for k in k_values: out[f"T@{k}"]=trustworthiness_from_ranks(ho,hr,lo,k); out[f"C@{k}"]=continuity_from_ranks(ho,lo,lr,k)
    return out
def pareto_frontier(df,metric_columns,atol=1e-12):
    v=df[metric_columns].to_numpy(float); keep=np.ones(len(df),bool)
    for i in range(len(df)):
        for j in range(len(df)):
            if i!=j and np.all(v[j]>=v[i]-atol) and np.any(v[j]>v[i]+atol): keep[i]=False; break
    return df.loc[keep].copy().reset_index(drop=True)
if RUN_SCREENING:
    Dh=squared_euclidean_distances(X_screen); high_order_screen,high_ranks_screen=neighbor_order_and_ranks(Dh)
    # identity test
    for k in [5,10,20,50]: assert np.isclose(trustworthiness_from_ranks(high_order_screen,high_ranks_screen,high_order_screen,k),1) and np.isclose(continuity_from_ranks(high_order_screen,high_order_screen,high_ranks_screen,k),1)
    rows=[]
    for p,Z in screening_embeddings.items(): row={"Perplexity":p}; row.update(neighborhood_quality(Z,high_order_screen,high_ranks_screen)); rows.append(row)
    neighborhood_quality_df=pd.DataFrame(rows).sort_values("Perplexity").reset_index(drop=True); metric_cols=[c for c in neighborhood_quality_df if c.startswith("T@") or c.startswith("C@")]; neighborhood_quality_df["Mean Trustworthiness"]=neighborhood_quality_df[[c for c in metric_cols if c.startswith("T@")]].mean(1); neighborhood_quality_df["Mean Continuity"]=neighborhood_quality_df[[c for c in metric_cols if c.startswith("C@")]].mean(1); neighborhood_quality_df["Worst Neighborhood Score"]=neighborhood_quality_df[metric_cols].min(1)
    pareto_candidates_df=pareto_frontier(neighborhood_quality_df,metric_cols); ps=pareto_candidates_df.sort_values("Worst Neighborhood Score",ascending=False)
    if len(ps)>=2: provisional_shortlist_df=ps.head(3).copy()
    else:
        rem=neighborhood_quality_df[~neighborhood_quality_df.Perplexity.isin(ps.Perplexity)].sort_values("Worst Neighborhood Score",ascending=False); provisional_shortlist_df=pd.concat([ps,rem.head(2-len(ps))],ignore_index=True)
    P11_CANDIDATE_PERPLEXITIES=provisional_shortlist_df.Perplexity.astype(int).tolist(); neighborhood_quality_df.to_csv(TABLE_DIR/"perplexity_neighborhood_quality.csv",index=False); provisional_shortlist_df.to_csv(TABLE_DIR/"p10_provisional_candidate_shortlist.csv",index=False)
    display(neighborhood_quality_df); print("P11 candidates:",P11_CANDIDATE_PERPLEXITIES); print("CHECKPOINT 10 — Multi-scale Neighborhood Evaluation: PASS ✅")
else: print("P10 READY — depends on P09 screening outputs.")

P10 READY — depends on P09 screening outputs.


## P11 — Multi-seed stability

### AI PROMPTING LOG

> Với 2–3 candidate perplexities từ P10, build P một lần mỗi perplexity rồi optimize seeds [0,42,123]. Tính T/C mỗi seed và pairwise kNN overlap giữa seed embeddings ở nhiều k. Không dùng coordinate MSE. Aggregate mean/std/worst; chọn PRIMARY + ALTERNATIVE bằng robust neighborhood + stability, runtime chỉ tie-break. Không gọi global optimum.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [13]:
from itertools import combinations
def knn_indices_from_distances(D,k):
    W=D.copy(); np.fill_diagonal(W,np.inf); return np.argsort(W,axis=1,kind="mergesort")[:,:k]
def neighborhood_overlap_score(a,b):
    vals=np.array([len(set(x).intersection(z))/a.shape[1] for x,z in zip(a,b)],float); return float(vals.mean()),vals
if RUN_FULL:
    P11_CANDIDATES=[int(p) for p in P11_CANDIDATE_PERPLEXITIES]; P11_SEEDS=[0,42,123]; K_VALUES=[5,10,20,50]; p11_embeddings={}; quality_rows=[]
    for p in P11_CANDIDATES:
        cr=conditional_probability_matrix(Dh,p); assert cr["converged"].all(); Pp=symmetrize_probabilities(cr["P_cond"]); p11_embeddings[p]={}
        for seed in P11_SEEDS:
            t=time.perf_counter(); o=optimize_tsne_embedding(Pp,n_iter=750,early_exaggeration_iter=250,momentum_switch_iter=250,patience=120,random_state=seed); Z=o["embedding"].copy(); runtime=time.perf_counter()-t; metrics=neighborhood_quality(Z,high_order_screen,high_ranks_screen,K_VALUES); row={"Perplexity":p,"Seed":seed,"Runtime Seconds":runtime,"Initial KL":o["initial_kl"],"Final KL":o["kl_divergence"],"Best KL":o["best_kl_divergence"],"Best Iteration":o["best_iteration"],"Actual Iterations":o["n_iter"],"Stopped Early":o["stopped_early"]}; row.update(metrics); quality_rows.append(row); p11_embeddings[p][seed]=Z; pd.DataFrame({"original_index":screen_indices,"TSNE_1":Z[:,0],"TSNE_2":Z[:,1]}).to_csv(EMBEDDING_DIR/f"p11_p{p}_seed{seed}.csv",index=False)
        del Pp,cr; gc.collect()
    seed_quality_results_df=pd.DataFrame(quality_rows).sort_values(["Perplexity","Seed"]).reset_index(drop=True); seed_quality_results_df.to_csv(TABLE_DIR/"p11_multiseed_quality.csv",index=False)
    stability_rows=[]
    for p in P11_CANDIDATES:
        for a,b in combinations(P11_SEEDS,2):
            for k in K_VALUES:
                na=knn_indices_from_distances(squared_euclidean_distances(p11_embeddings[p][a]),k); nb=knn_indices_from_distances(squared_euclidean_distances(p11_embeddings[p][b]),k); mean,vals=neighborhood_overlap_score(na,nb); stability_rows.append({"Perplexity":p,"Seed A":a,"Seed B":b,"k":k,"Mean kNN Overlap":mean,"Min Sample Overlap":vals.min(),"Median Sample Overlap":np.median(vals)})
    seed_stability_df=pd.DataFrame(stability_rows); seed_stability_df.to_csv(TABLE_DIR/"p11_pairwise_seed_stability.csv",index=False)
    metric_cols=[f"T@{k}" for k in K_VALUES]+[f"C@{k}" for k in K_VALUES]; rob=[]
    for p in P11_CANDIDATES:
        g=seed_quality_results_df[seed_quality_results_df.Perplexity==p]; s=seed_stability_df[seed_stability_df.Perplexity==p]; allv=g[metric_cols].to_numpy(float); rob.append({"Perplexity":p,"Mean Neighborhood Score":allv.mean(),"Robust Worst Neighborhood":allv.min(),"Mean Seed Stability":s["Mean kNN Overlap"].mean(),"Worst Seed Stability":s["Mean kNN Overlap"].min(),"Runtime Mean":g["Runtime Seconds"].mean(),"Runtime Std":g["Runtime Seconds"].std(ddof=0),"Final KL Mean":g["Final KL"].mean(),"Final KL Std":g["Final KL"].std(ddof=0)})
    candidate_robustness_df=pd.DataFrame(rob); candidate_robustness_df.to_csv(TABLE_DIR/"p11_candidate_robustness_summary.csv",index=False)
    select_cols=["Robust Worst Neighborhood","Mean Neighborhood Score","Worst Seed Stability","Mean Seed Stability"]; robust_pareto_df=pareto_frontier(candidate_robustness_df,select_cols); pool=robust_pareto_df.sort_values(["Robust Worst Neighborhood","Worst Seed Stability","Mean Neighborhood Score","Mean Seed Stability","Runtime Mean"],ascending=[False,False,False,False,True]).reset_index(drop=True); primary=pool.iloc[0]; PRIMARY_PERPLEXITY=int(primary.Perplexity)
    if len(pool)>=2: alt=pool.iloc[1]
    else: alt=candidate_robustness_df[candidate_robustness_df.Perplexity!=PRIMARY_PERPLEXITY].sort_values(["Robust Worst Neighborhood","Worst Seed Stability","Mean Neighborhood Score","Mean Seed Stability","Runtime Mean"],ascending=[False,False,False,False,True]).iloc[0]
    ALTERNATIVE_PERPLEXITY=int(alt.Perplexity); final_selection_df=pd.DataFrame([{"Role":"PRIMARY","Perplexity":PRIMARY_PERPLEXITY},{"Role":"ALTERNATIVE","Perplexity":ALTERNATIVE_PERPLEXITY}]); final_selection_df.to_csv(TABLE_DIR/"p11_primary_alternative_selection.csv",index=False); display(candidate_robustness_df); display(final_selection_df); print("CHECKPOINT 11 — Multi-seed Stability & Robust Selection: PASS ✅")
else: print("P11 READY — set EXECUTION_PROFILE='full'.")

P11 READY — set EXECUTION_PROFILE='full'.


## P12 — Final full-data run

### AI PROMPTING LOG

> Khóa PRIMARY_PERPLEXITY từ P11 và chạy exact t-SNE trên full 5620×64. Trước chạy in memory preflight; build D→P rồi giải phóng D/P_cond trước optimization. Labels chỉ post-hoc. Xuất final embedding CSV, figures, config, run summary, optimization history, reproducibility manifest. Không tune lại.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [14]:
if RUN_FULL:
    FINAL_PERPLEXITY=int(PRIMARY_PERPLEXITY); FINAL_RANDOM_STATE=42; n=len(X); print("N² elements:",n*n,"one float64 matrix MiB:",n*n*8/1024**2)
    t0=time.perf_counter(); D_full=squared_euclidean_distances(X); cr_full=conditional_probability_matrix(D_full,FINAL_PERPLEXITY); assert cr_full["converged"].all(); P_cond_full=cr_full["P_cond"]; P_full=symmetrize_probabilities(P_cond_full); achieved_full=cr_full["perplexities"].copy(); del D_full,P_cond_full,cr_full; gc.collect()
    optimizer_t0=time.perf_counter(); final_optimization=optimize_tsne_embedding(P_full,learning_rate=200.0,n_iter=1000,early_exaggeration=12.0,early_exaggeration_iter=250,initial_momentum=0.5,final_momentum=0.8,momentum_switch_iter=250,patience=150,random_state=FINAL_RANDOM_STATE,verbose=True); optimizer_seconds=time.perf_counter()-optimizer_t0; final_runtime_seconds=time.perf_counter()-t0
    final_embedding=final_optimization["embedding"].copy(); assert final_embedding.shape==(5620,2) and np.isfinite(final_embedding).all() and np.isclose(final_optimization["kl_divergence"],final_optimization["best_kl_divergence"],rtol=1e-8,atol=1e-10)
    FINAL_TAG=f"p{FINAL_PERPLEXITY}"; final_embedding_df=pd.DataFrame({"original_index":np.arange(len(X)),"TSNE_1":final_embedding[:,0],"TSNE_2":final_embedding[:,1],"label":y}); final_embedding_df.to_csv(EMBEDDING_DIR/f"final_optdigits_embedding_{FINAL_TAG}.csv",index=False)
    final_config_df=pd.DataFrame([{"Parameter":"Samples","Value":len(X)},{"Parameter":"Original dimensions","Value":X.shape[1]},{"Parameter":"Embedding dimensions","Value":2},{"Parameter":"Perplexity","Value":FINAL_PERPLEXITY},{"Parameter":"Learning rate","Value":200.0},{"Parameter":"Max iterations","Value":1000},{"Parameter":"Early exaggeration","Value":12.0},{"Parameter":"Early exaggeration iterations","Value":250},{"Parameter":"Initial momentum","Value":0.5},{"Parameter":"Final momentum","Value":0.8},{"Parameter":"Random state","Value":FINAL_RANDOM_STATE},{"Parameter":"Implementation","Value":"Exact NumPy t-SNE from scratch"}]); final_config_df.to_csv(TABLE_DIR/f"final_configuration_{FINAL_TAG}.csv",index=False)
    final_run_summary_df=pd.DataFrame([{"Samples":len(X),"Features":X.shape[1],"Perplexity":FINAL_PERPLEXITY,"Initial KL":final_optimization["initial_kl"],"Best KL":final_optimization["best_kl_divergence"],"Final Restored KL":final_optimization["kl_divergence"],"Best Iteration":final_optimization["best_iteration"],"Actual Iterations":final_optimization["n_iter"],"Stopped Early":final_optimization["stopped_early"],"Runtime Seconds":final_runtime_seconds,"Optimizer Seconds":optimizer_seconds,"Achieved Perplexity Mean":achieved_full.mean(),"Achieved Perplexity Min":achieved_full.min(),"Achieved Perplexity Max":achieved_full.max(),"Achieved Perplexity Max Error":np.max(np.abs(achieved_full-FINAL_PERPLEXITY))}]); final_run_summary_df.to_csv(TABLE_DIR/f"final_run_summary_{FINAL_TAG}.csv",index=False)
    hist=pd.DataFrame({"Iteration":np.arange(1,len(final_optimization["kl_history"])+1),"KL":final_optimization["kl_history"],"Gradient Norm":final_optimization["gradient_norm_history"]}); hist.to_csv(TABLE_DIR/f"final_optimization_history_{FINAL_TAG}.csv",index=False)
    print("Evaluating final full-data neighborhood preservation..."); D_eval=squared_euclidean_distances(X); full_order,full_ranks=neighbor_order_and_ranks(D_eval); del D_eval; gc.collect(); fq=neighborhood_quality(final_embedding,full_order,full_ranks,K_VALUES); del full_order,full_ranks; gc.collect()
    final_quality_df=pd.DataFrame([{"k":k,"Trustworthiness":fq[f"T@{k}"],"Continuity":fq[f"C@{k}"]} for k in K_VALUES]); final_quality_df.to_csv(TABLE_DIR/f"final_neighborhood_quality_{FINAL_TAG}.csv",index=False)
    pd.DataFrame([{"Perplexity":FINAL_PERPLEXITY,"Samples":len(X),"Mean Trustworthiness":final_quality_df["Trustworthiness"].mean(),"Mean Continuity":final_quality_df["Continuity"].mean(),"Worst Neighborhood Score":final_quality_df[["Trustworthiness","Continuity"]].to_numpy().min()}]).to_csv(TABLE_DIR/f"final_neighborhood_quality_summary_{FINAL_TAG}.csv",index=False)
    plt.figure(figsize=(10,8));
    for digit in range(10):
        m=y==digit; plt.scatter(final_embedding[m,0],final_embedding[m,1],s=9,alpha=.7,label=str(digit))
    plt.title(f"Optdigits — t-SNE From Scratch | Perplexity={FINAL_PERPLEXITY}"); plt.legend(title="Digit",ncol=2); plt.grid(alpha=.15); plt.savefig(FIGURE_DIR/f"final_optdigits_tsne_{FINAL_TAG}.png",dpi=200,bbox_inches="tight"); plt.show()
    plt.figure(figsize=(9,4)); plt.plot(np.arange(1,len(final_optimization["kl_history"])+1),final_optimization["kl_history"]); plt.axvline(250,ls="--"); plt.xlabel("Iteration"); plt.ylabel("KL(P||Q)"); plt.savefig(FIGURE_DIR/f"final_kl_convergence_{FINAL_TAG}.png",dpi=170,bbox_inches="tight"); plt.show()
    plt.figure(figsize=(9,4)); plt.plot(np.arange(1,len(final_optimization["gradient_norm_history"])+1),final_optimization["gradient_norm_history"]); plt.axvline(250,ls="--"); plt.xlabel("Iteration"); plt.ylabel("Gradient norm"); plt.savefig(FIGURE_DIR/f"final_gradient_norm_{FINAL_TAG}.png",dpi=170,bbox_inches="tight"); plt.show()
    manifest={"project":"t-SNE From Scratch on UCI Optdigits","n_samples":5620,"n_features":64,"primary_perplexity":PRIMARY_PERPLEXITY,"alternative_perplexity":ALTERNATIVE_PERPLEXITY,"final_random_state":FINAL_RANDOM_STATE,"final_kl":float(final_optimization["kl_divergence"]),"runtime_seconds":final_runtime_seconds,"optimizer_seconds":optimizer_seconds,"final_neighborhood_quality":{**{f"T@{k}":float(fq[f"T@{k}"]) for k in K_VALUES},**{f"C@{k}":float(fq[f"C@{k}"]) for k in K_VALUES}},"label_usage":"not used in fitting; only stratified screening and post-hoc visualization"}; (OUTPUT_ROOT/"reproducibility_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")
    print("CHECKPOINT 12 — Final Full Optdigits t-SNE: PASS ✅")
else: print("P12 READY — exact full run intentionally gated behind EXECUTION_PROFILE='full'.")


P12 READY — exact full run intentionally gated behind EXECUTION_PROFILE='full'.


## P13 — Reproducibility and handoff

### AI PROMPTING LOG

> Không thay đổi thuật toán. Audit required state/artifacts, tạo phase status, AI prompting log summary, README, requirements, structure, final results summary, expected-artifact contract và ZIP. Phân biệt artifact đã tạo với artifact chỉ sinh sau heavy run. Restart/Run-all phải không phụ thuộc hidden state. Không mang lại legacy index leakage hay hard-coded perplexity mismatch.

**Logic cần giữ nguyên:** input/state → operation → validation → output → checkpoint.

In [15]:
status={"execution_profile":EXECUTION_PROFILE,"core_P00_P08":"executed","P09_P10":"executed" if RUN_SCREENING else "not executed","P11_P12":"executed" if RUN_FULL else "not executed"}
(OUTPUT_ROOT/"run_status.json").write_text(json.dumps(status,indent=2),encoding="utf-8")
print("Package handoff includes README, 14 prompt files, master prompt log, prompt-code mapping, source, tests, raw+clean data, audit artifacts, templates and runtime artifact contract.")
print("Run scripts/audit_package.py before submission.")
print("CHECKPOINT 13 — Handoff structure PASS; empirical checkpoints remain conditional on chosen execution profile. ✅")

Package handoff includes README, 14 prompt files, master prompt log, prompt-code mapping, source, tests, raw+clean data, audit artifacts, templates and runtime artifact contract.
Run scripts/audit_package.py before submission.
CHECKPOINT 13 — Handoff structure PASS; empirical checkpoints remain conditional on chosen execution profile. ✅
